[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_13/MFM_ch13_feature_importance/MFM_ch13_feature_importance.ipynb)

# MFM_ch13_feature_importance

**Modelling Financial Markets (MFM) — Chapter 13: Machine Learning in Finance**

Feature importance for S&P 500 direction models: in-sample MDI of a random forest, out-of-sample MDA (permutation, change in AUC) and mean absolute SHAP values of a gradient boosting model.

*Author: Daniel Traian Pele*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_13'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip -q install shap

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## Style and data

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'

SEED = 42


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


In [ ]:
import os
DATA_URL = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
SERIES = {'sp500': 'GSPC.INDX', 'vix': 'VIX.INDX', 'btc': 'BTC-USD.CC'}
START = {'sp500': '2000-01-01', 'vix': '2000-01-01', 'btc': '2014-09-17'}


def load_data(name='sp500', start=None, end='2026-09-18'):
    """OHLCV zilnic din data/market: copia locala, altfel fisierul din repo-ul GitHub MFM."""
    fname = SERIES[name] + '.csv'
    local = [os.path.join(d, 'data', 'market', fname) for d in ('.', '..', '../..', '../../..')]
    path = next((p for p in local if os.path.exists(p)), DATA_URL + fname)
    df = pd.read_csv(path, parse_dates=['date']).set_index('date')
    df = df.rename(columns=str.capitalize)[['Open', 'High', 'Low', 'Close', 'Volume']]
    return df.loc[start or START[name]:end]


spx = load_data('sp500')['Close']
vix = load_data('vix')['Close']
btc = load_data('btc')['Close']
log_spx = np.log(spx)
print('S&P 500:', spx.index[0].date(), '->', spx.index[-1].date(), len(spx), 'obs')
print('BTC    :', btc.index[0].date(), '->', btc.index[-1].date(), len(btc), 'obs')

## Helper functions

In [ ]:
H_LABEL = 5

def ffd_weights(d, threshold=1e-4, max_size=10_000):
    """Ponderile w_k = -w_{k-1} (d - k + 1) / k, trunchiate cand |w_k| < threshold."""
    w = [1.0]
    for k in range(1, max_size):
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < threshold:
            break
        w.append(w_k)
    return np.array(w)


def frac_diff_ffd(series, d, threshold=1e-4):
    """Seria diferentiata fractionar cu fereastra fixa: X~_t = sum_k w_k X_{t-k}."""
    w = ffd_weights(d, threshold)
    width = len(w)
    x = series.dropna().values
    out = np.convolve(x, w, mode='valid')          # out[j] = sum_k w_k x[j+width-1-k]
    return pd.Series(out, index=series.dropna().index[width - 1:])


def rsi(close, n=14):
    delta = close.diff()
    up = delta.clip(lower=0).ewm(alpha=1 / n).mean()
    dn = (-delta.clip(upper=0)).ewm(alpha=1 / n).mean()
    return 100 - 100 / (1 + up / dn)


def build_features(close, vix=None, d_ffd=None):
    """Caracteristici calculate doar cu informatie disponibila la momentul t."""
    lp = np.log(close)
    r = lp.diff()
    f = pd.DataFrame(index=close.index)
    for h in (1, 5, 20, 60, 120):
        f[f'mom_{h}'] = lp.diff(h)
    f['vol_20'] = r.rolling(20).std() * np.sqrt(252)
    f['vol_60'] = r.rolling(60).std() * np.sqrt(252)
    f['vol_ratio'] = f['vol_20'] / f['vol_60']
    f['dist_ma50'] = lp - lp.rolling(50).mean()
    f['dist_ma200'] = lp - lp.rolling(200).mean()
    f['rsi_14'] = rsi(close) / 100
    if vix is not None:
        v = np.log(vix.reindex(close.index).ffill())
        f['vix_level'] = v
        f['vix_chg_5'] = v.diff(5)
    if d_ffd is not None:
        f['ffd_price'] = frac_diff_ffd(lp, d_ffd)
    return f


class PurgedKFold:
    """
    K-Fold fara amestecare pentru etichete care se suprapun in timp.

    t1 : pd.Series indexata dupa momentul observatiei t0, cu valoarea = momentul
         in care se cunoaste eticheta (sfarsitul ferestrei etichetei).
    Purjare: se elimina din antrenare observatiile al caror interval [t0, t1]
             se suprapune cu intervalul setului de test.
    Embargo: se elimina si o fractiune pct_embargo de observatii imediat dupa test.
    """

    def __init__(self, n_splits=5, t1=None, pct_embargo=0.01):
        self.n_splits = n_splits
        self.t1 = t1
        self.pct_embargo = pct_embargo

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        idx = np.arange(len(X))
        t0 = self.t1.index.values
        t1 = self.t1.values
        embargo = int(len(X) * self.pct_embargo)
        for test in np.array_split(idx, self.n_splits):
            start, stop = test[0], test[-1]
            test_t0, test_t1 = t0[start], t1[test].max()
            # purjare: antrenare doar pe observatii care nu se suprapun cu testul
            before = idx[(t1 < test_t0)]
            after_start = min(stop + 1 + embargo, len(X))
            after = idx[after_start:][t0[after_start:] > test_t1]
            yield np.concatenate([before, after]), test


def find_min_d(logp, grid=np.round(np.arange(0, 1.01, 0.05), 2), crit_level='5%'):
    """Cel mai mic d pentru care seria FFD este stationara (ADF la 5%)."""
    rows = []
    for d in grid:
        x = frac_diff_ffd(logp, d).dropna()
        adf = adfuller(x, maxlag=1, regression='c', autolag=None)
        corr = np.corrcoef(logp.loc[x.index], x)[0, 1]
        rows.append((d, adf[0], adf[4][crit_level], corr))
    out = pd.DataFrame(rows, columns=['d', 'adf', 'crit', 'corr']).set_index('d')
    d_star = out.index[out['adf'] < out['crit']].min()
    return out, d_star


def modelling_frame(d_star):
    feats = build_features(spx, vix=vix, d_ffd=d_star)
    fwd = np.log(spx).shift(-H_LABEL) - np.log(spx)
    df = feats.assign(fwd=fwd).dropna()
    df = df.loc['2001-01-01':]
    df['y'] = (df['fwd'] > 0).astype(int)
    pos = np.searchsorted(spx.index, df.index)
    df['t1'] = spx.index[np.minimum(pos + H_LABEL, len(spx) - 1)]
    return df

## Charts

In [ ]:
def fig_feature_importance(df):
    feat_cols = [c for c in df.columns if c not in ('fwd', 'y', 't1')]
    X, y = df[feat_cols], df['y'].values
    split = int(len(df) * 0.7)
    gap = split + H_LABEL + int(0.01 * len(df))
    rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=100, max_features='sqrt',
                                n_jobs=-1, random_state=SEED).fit(X.iloc[:split], y[:split])
    mdi = pd.Series(rf.feature_importances_, index=feat_cols)
    mda = pd.Series(permutation_importance(rf, X.iloc[gap:], y[gap:], scoring='roc_auc',
                                           n_repeats=20, random_state=SEED, n_jobs=-1).importances_mean,
                    index=feat_cols)
    try:
        import shap
        gbm = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.03, max_iter=200,
                                             min_samples_leaf=100, random_state=SEED)
        gbm.fit(X.iloc[:split], y[:split])
        sample = X.iloc[gap:].sample(min(800, len(X) - gap), random_state=SEED)
        explainer = shap.Explainer(gbm.predict_proba, X.iloc[:split].sample(200, random_state=SEED))
        sv = explainer(sample).values[..., 1]
        shap_imp = pd.Series(np.abs(sv).mean(0), index=feat_cols)
    except Exception as e:
        print('   SHAP skipped:', e)
        shap_imp = None

    order = mdi.sort_values().index
    n_panels = 3 if shap_imp is not None else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(7.4, 3.3), sharey=True)
    axes[0].barh(order, mdi[order], color=MainBlue, alpha=0.85)
    axes[0].set_title('MDI (in-sample)', fontsize=8.5, loc='left')
    axes[1].barh(order, mda[order], color=np.where(mda[order] > 0, Forest, IDAred), alpha=0.85)
    axes[1].axvline(0, color=Gray, lw=0.5)
    axes[1].set_title('MDA (OOS, $\\Delta$AUC)', fontsize=8.5, loc='left')
    if shap_imp is not None:
        axes[2].barh(order, shap_imp[order], color=Amber, alpha=0.85)
        axes[2].set_title('mean |SHAP| (OOS)', fontsize=8.5, loc='left')
    for ax in axes:
        ax.tick_params(axis='y', labelsize=7.5)
        ax.tick_params(axis='x', labelsize=7)
    plt.tight_layout()
    save_fig('ch13_feature_importance')
    return mdi, mda, shap_imp

## Run

In [ ]:
_, d_star = find_min_d(log_spx)
print('d* =', d_star)
df = modelling_frame(d_star)
print('modelling frame:', df.shape)
mdi, mda, shp = fig_feature_importance(df)
print(pd.DataFrame({'MDI': mdi, 'MDA': mda, 'SHAP': shp}).round(4))

![ch13_feature_importance](./ch13_feature_importance.png)

Output: `ch13_feature_importance.pdf`